In [ ]:
%pip install -e ../../../
%pip install scipy
%pip install plotly
%pip install nbformat

In [ ]:
# Setup
from numpy.random import default_rng
RNG = default_rng(12820)

import pandas as pd
static_properties_path = "../water_utilities-static_properties.xlsx"
water_utilities_df = pd.read_excel(static_properties_path, sheet_name='entities')
UTILITIES_IDS = list(water_utilities_df['water_utility_id'].astype(str))

## Funds balance

In [ ]:
balance_df = pd.DataFrame(
    data=[[0.0]*len(water_utilities_df)],
    columns=UTILITIES_IDS,
    index=pd.DatetimeIndex([pd.to_datetime("2000-01-01")], name='timestamp')
)

## Water price

We start from the 2022 Vewin report. 
At the end of chapter 2 there are the water prices for each water utility in 2021.

So we use inflation to get them in 2000.

### Fixed component

In [ ]:
from scipy.stats import skewnorm
import numpy as np
Y2021 = pd.to_datetime("2021-01-01")
FIXED_2021 = [76, 70, 64, 54, 76, 60, 75, 70, 70, 88]
FIXED_MEAN_2021 = np.mean(FIXED_2021)
FIXED_STD_2021 = np.std(FIXED_2021)

fixed_p_df = pd.DataFrame(
    data=skewnorm.rvs(
        a=5,  # How much positive skewness
        loc=FIXED_MEAN_2021,
        scale=FIXED_STD_2021,
        size=(1, len(water_utilities_df)),
        random_state=RNG
    ),
    columns=UTILITIES_IDS,
    index=pd.DatetimeIndex([Y2021], name='timestamp')
)

### Variable component

Also taken from the Vewin report (0.75 in 2021)

In [ ]:
VARIAB_2021 = [0.63, 0.64, 1.22, 0.83, 1.0, 0.726, 0.859, 0.46, 0.782]
VARIAB_MEAN_2021 = np.mean(VARIAB_2021)
VARIAB_STD_2021 = np.std(VARIAB_2021)

variab_p_df = pd.DataFrame(
    data=skewnorm.rvs(
        a=5, # How much positive skewness
        loc=VARIAB_MEAN_2021,
        scale=VARIAB_STD_2021,
        size=(1,len(water_utilities_df)),
        random_state=RNG
    ),
    columns=UTILITIES_IDS,
    index=pd.DatetimeIndex([Y2021], name='timestamp')
)

## Water price - Selling price 

We assume that the price for other water utilities could be between 30% and 50% to what is for consumers.

In [ ]:
sell_p_df = variab_p_df*RNG.uniform(low=0.3, high=0.5, size=(1, len(water_utilities_df)))

## Saving

First we need to get the prices in 2000.
We do it by inverting inflation

In [ ]:
# Prepare water prices 
# Upload inflation data 
from water_futures_battle.economy.dynamic_properties import EconomyDB

econ_db = EconomyDB.load_from_file('../../economy/economy-dynamic_properties.xlsx')
inflation = econ_db[EconomyDB.INFLATION]['NL0000']

def deflate_prices(costs: pd.Series, base_year: pd.Timestamp, inflation: pd.Series) -> pd.DataFrame:
    
    cum_inflation = (1 + inflation/100).cumprod()

    base_index = cum_inflation.loc[base_year]

    deflated_values = costs.values / (base_index / cum_inflation.values[:, None])
    
    return pd.DataFrame(
        data=deflated_values,
        columns=costs.index,
        index=cum_inflation.index
    )

from water_futures_battle.core.base_model.properties import DynamicProperties
wu_dps = DynamicProperties('water_utilities-dynamic_properties')
from water_futures_battle.water_utilities.dynamic_properties import WaterUtilityDB

Y2000 = pd.to_datetime("2000-01-01")
wu_dps[WaterUtilityDB.BALANCE] = balance_df.loc[[Y2000]].copy()

Y1999 = pd.to_datetime("1999-01-01")
for property, df in zip(
    [WaterUtilityDB.WPRICE_FIXED,WaterUtilityDB.WPRICE_VARIA,WaterUtilityDB.WPRICE_SELL],
    [fixed_p_df, variab_p_df, sell_p_df]
):
    full_df = deflate_prices(df.loc[Y2021], Y2021, inflation)

    wu_dps[property] = full_df.loc[[Y1999]].copy()


wu_dps.dump('..')